In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# ===============================
# LOAD DATA
# ===============================
file_path = "production data with category.xlsx"
df = pd.read_excel(file_path)

# ===============================
# LABEL CREATION
# ===============================
df["Smart_Decision"] = (df["Production Gap"] <= 0).astype(int)

# ===============================
# CLEAN & FEATURE ENGINEERING
# ===============================

# Encode Category
df["Category_enc"] = df["Category"].map({
    "Runner": 0,
    "Repeater": 1,
    "Stranger": 2
})

# Encode Shift
df["Shift_enc"] = df["Shift"].map({"A": 0, "B": 1})

# Encode Machine
le_machine = LabelEncoder()
df["Machine_enc"] = le_machine.fit_transform(df["Machine ID"])

# Convert cycle times (example: "72/02" → 36)
def parse_cycle(x):
    if isinstance(x, str) and "/" in x:
        a, b = x.split("/")
        return float(a) / float(b)
    return np.nan

df["STD_Cycle"] = df["STD. Cycle Time"].apply(parse_cycle)
df["ACT_Cycle"] = df["Actual Standard Cycle Time"].apply(parse_cycle)

# ===============================
# SELECT FEATURES
# ===============================
features = [
    "Machine_enc",
    "Category_enc",
    "Shift_enc",
    "Required Qty",
    "Part Per Hour",
    "Hours Required",
    "Setup Rejections",
    "Actual Rejections",
    "STD_Cycle",
    "ACT_Cycle"
]

X = df[features].fillna(0)
y = df["Smart_Decision"]

# ===============================
# TRAIN / TEST SPLIT
# ===============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# ===============================
# TRAIN MODEL
# ===============================
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)

# ===============================
# EVALUATION
# ===============================
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
